# 03. Multi-output 단일 모델 학습 (Walk-Forward)

## 📋 개요
이 노트북은 **타깃 중심 정렬(Target-Centric Alignment)** 방식을 사용하여 단일 모델(LightGBM, RandomForest, MLP)을 학습합니다.
- **핵심 논리**: $t$ 시점의 가격을 맞추기 위해 $t-i$ 시점의 데이터를 참조하는 Direct Forecasting 전략입니다.
- **Walk-Forward 검증**: 시간의 흐름에 따라 훈련/검증/테스트 윈도우를 이동시키며 모델의 강건성을 평가합니다. (Look-ahead 편향을 막기 위한 Embargo Gap 자동 적용)

## 🔧 최근 업데이트 내역
- **v3.10.0**: 사다리꼴 역산 로직을 `src/utils/trapezoidal.py`의 `trapezoid_log_close()`로 교체하여 중복 수식 제거.
- **v3.9.2**: `save_model_artifact()` 호출부에 `hyperparameters` 키 명시 (param_hash 충돌 방지).
- **v3.9.1**: `log_return_1d` 모드 평가 지표 산출 시 **사다리꼴 적분 보정(Trapezoidal Rule)** 적용.
- **v3.9.0**: 누적 로그 수익률 폐기 및 1일 당일 등락률(`target_log_return_1d`) 타겟 모드 신설.
- **v3.8.0**: MLP 모델 신설 및 단일 모델 동적 로딩, 앙상블(`+` 기호) 설정 시 조기 차단 로직 추가.

## 🔧 Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
import copy
from tqdm import tqdm

from src.utils.config import load_config, ProjectPaths, is_ensemble
from src.utils.trapezoidal import trapezoid_log_close
from src.modeling.trainer import WalkForwardTrainer
from src.models.artifact import save_model_artifact

warnings.filterwarnings('ignore')

In [ ]:
# ── [config 셀] ──────────────────────────────────────────────
cfg       = load_config()
train_cfg = cfg['training']

active_model_str = cfg.get('active_model', 'lightgbm')
if is_ensemble(active_model_str):
    raise ValueError(
        f"active_model='{active_model_str}'은 앙상블 조합입니다.\n"
        f"03단계는 단일 모델 전용입니다. 앙상블은 03b를 사용하세요."
    )

paths = ProjectPaths.from_config(cfg)
paths.ensure_dirs()

print(f"🚀 [Step 3] Multi-output 학습 시작")
print(f"   - 기준일 : {paths.reference_date}")
print(f"   - 모델   : {paths.folder_name}")
print(f"\n📂 사용 경로:")
print(f"   - 입력: {paths.get_dataset_parquet()}")
print(f"   - 모델: {paths.get_model_dir()}")

## 1️⃣ 데이터 로드
02단계에서 생성된 통합 Feature 데이터셋을 로드합니다. 개별 시점의 피처 시프트는 Trainer 내부에서 타깃별로 수행됩니다.

In [ ]:
print("📥 Loading dataset...")
df = pd.read_parquet(paths.get_dataset_parquet())
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

# ✨ v3.10.0: feature_risk_composite가 feature_ 접두사를 갖게 되어 예외 처리 불필요
feature_cols = [c for c in df.columns if c.startswith('feature_')]

print(f"   - 학습 데이터 행수: {len(df):,}")
print(f"   - 사용 피처 수: {len(feature_cols)}")

## 2️⃣ 모델 및 Trainer 초기화
`config.yaml`의 `active_model` 설정에 따라 최적의 Multi-output 모델 클래스를 동적으로 초기화합니다.
- **LightGBM**: 타깃($h_1 \sim h_5$)별 독립 Booster 운영
- **RandomForest**: `MultiOutputRegressor` 래퍼 기반 학습
- **MLP**: 타깃을 동시 출력하는 단일 신경망 기반 공유 잠재 표현(Shared Representation) 학습

In [ ]:
model_type = active_model_str

if model_type in ("lightgbm", "lgbm"):
    print("🤖 Initializing LightGBM Model...")
    from src.models.lightgbm_model import LightGBMModel
    model = LightGBMModel(
        model_version=f"v1_lgbm_{paths.reference_date}",
        params=train_cfg['lgbm_params'],
        feature_list=feature_cols,
        categorical_features=[]
    )
    fit_kwargs = {
        'num_boost_round': 1000,
        'callbacks': [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
    }

elif model_type in ("randomforest", "rf"):
    print("🌲 Initializing Random Forest Multi-output Model...")
    from src.models.randomforest_model import RandomForestMultiModel
    model = RandomForestMultiModel(
        model_version=f"v1_rf_{paths.reference_date}",
        params=train_cfg["randomforest_params"],
        feature_list=feature_cols
    )
    fit_kwargs = {}

elif model_type == "mlp":
    print("🧠 Initializing MLP Multi-output Model...")
    from src.models.mlp_model import MLPModel
    model = MLPModel(
        model_version=f"v1_mlp_{paths.reference_date}",
        params=train_cfg["mlp_params"],
        feature_list=feature_cols
    )
    fit_kwargs = {
        'epochs':   train_cfg["mlp_params"].get("epochs", 200),
        'patience': train_cfg["mlp_params"].get("patience", 15),
    }

else:
    from src.utils.config import resolve_model_name
    resolve_model_name(model_type)

horizons    = train_cfg.get('horizons', [1, 2, 3, 4, 5])
target_base = train_cfg.get('target_col_name', 'target_log_close')
model.target_columns = [f"{target_base}_h{h}" for h in horizons]

print("🔧 Initializing WalkForwardTrainer...")
trainer = WalkForwardTrainer(
    model=model,
    feature_cols=feature_cols,
    target_col_name=train_cfg.get('target_col_name', 'target_log_close'),
    target_type=train_cfg.get('target_type', 'log_close'),
    horizons=horizons,
    date_col='date'
)

## 3️⃣ Walk-Forward 학습 실행
각 시점($t$)의 가격을 정답으로 두고, $t-1, t-2, \dots, t-5$의 피처를 각각 매칭하여 5개의 내부 모델을 학습합니다.

In [ ]:
print("🏃 Running Target-Centric Walk-Forward Training...")

results = trainer.run(
    df=df,
    train_end=train_cfg['train_end'],
    valid_window_days=train_cfg['valid_window_days'],
    test_window_days=train_cfg['test_window_days'],
    fit_kwargs=fit_kwargs
)

## 4️⃣ 결과 저장 및 예측값 역산 (Inverse Transformation)

학습된 공간(예: Log Return)의 예측값을 실제 분석 가능한 원화(KRW) 가격으로 역산하여 저장합니다.

### 📐 타겟 모드별 역산 로직
**1. `log_close` 모드 (기본값)**
- $pred\_close(t+h) = \exp(pred\_log\_close(t+h))$

**2. `log_return_1d` 모드 (v3.9.1 / v3.10.0 모듈화)**
- 1일 당일 등락률 누적 오차 보정을 위해 **사다리꼴 적분(Trapezoidal Rule)**을 적용합니다.
- `src/utils/trapezoidal.py`의 `trapezoid_log_close()` 함수를 사용합니다.
- 수식: $y(t+h) = y(t) + \text{cumsum}_h + \frac{\Delta y(t) - \Delta y(t+h)}{2}$

In [ ]:
print("\n💾 Saving predictions & model artifact...")

val_pred_df  = results['val_predictions']
test_pred_df = results['test_predictions']
target_type  = results['target_type']

if target_type == "log_return_1d":
    close_ref = df[['date', 'ticker', 'close', 'target_log_return_1d']].drop_duplicates()

    for df_pred in [val_pred_df, test_pred_df]:
        df_merged = df_pred.merge(close_ref, on=['date', 'ticker'], how='left')
        log_close_base = np.log(df_merged['close'].clip(lower=1e-9))
        delta_y_t      = df_merged['target_log_return_1d'].values  # Δy(t) 앵커

        sorted_target_cols = sorted(results['target_cols'], key=lambda c: int(c.split('_h')[-1]))

        for idx, col in enumerate(sorted_target_cols):
            pred_col = f'pred_{col}'
            if pred_col not in df_merged.columns: continue

            pred_delta_h   = df_merged[pred_col].values  # Δy(t+h) 예측
            cum_log_return = sum(df_merged[f'pred_{c}'] for c in sorted_target_cols[:idx + 1])

            # ✨ v3.10.0: trapezoid_log_close() 모듈 사용
            pred_log_close = trapezoid_log_close(
                log_close_base, cum_log_return, delta_y_t, pred_delta_h
            )
            df_merged[f'pred_log_close_{col}'] = pred_log_close
            df_merged[f'pred_close_{col}']     = np.exp(pred_log_close)

        if df_pred is val_pred_df:
            val_pred_df = df_merged
        else:
            test_pred_df = df_merged

else:  # log_close 모드
    for df_pred in [val_pred_df, test_pred_df]:
        for col in results['target_cols']:
            pred_col = f'pred_{col}'
            if pred_col in df_pred.columns:
                df_pred[f'pred_close_{col}'] = np.exp(df_pred[pred_col])

# Parquet 저장
val_parquet_path  = paths.get_model_dir() / "val_predictions.parquet"
test_parquet_path = paths.get_model_dir() / "test_predictions.parquet"
val_pred_df.to_parquet(val_parquet_path,  index=False)
test_pred_df.to_parquet(test_parquet_path, index=False)

print(f"   - [Val]  Saved: {val_parquet_path}")
print(f"   - [Test] Saved: {test_parquet_path}")

# 성능 요약
print(f"\n📊 성능 요약  (target_type={target_type})")
print(f"   검증  Avg RMSE : {results['valid_metrics']['avg_rmse']:.6f}")
print(f"   테스트 Avg RMSE: {results['test_metrics']['avg_rmse']:.6f}")
print(f"\n   Horizon별 테스트 RMSE:")
for col, metrics in results['test_metrics']['per_horizon'].items():
    print(f"     {col}: RMSE={metrics['rmse']:.6f}, IC={metrics['ic_mean']:.4f}")

# CSV 및 아티팩트 저장
csv_pred_dir = paths.get_predictions_csv_dir()
csv_pred_dir.mkdir(exist_ok=True)
print(f"   - [Individual] Saving ticker CSVs to {csv_pred_dir}...")

try:
    df_master = pd.read_csv(paths.get_ticker_master())
    ticker_name_map = dict(zip(df_master['ticker'].astype(str), df_master['name']))
except Exception:
    ticker_name_map = {}

for ticker, group in tqdm(test_pred_df.groupby('ticker'), desc="Saving CSVs"):
    name = ticker_name_map.get(str(ticker), f"ticker_{ticker}")
    safe_name = str(name).replace('/', '_').replace('\\', '_')
    group.to_csv(csv_pred_dir / f"{safe_name}.csv", index=False, encoding='utf-8-sig')

model_save_dir = paths.get_model_dir()
save_model_artifact(
    model_name=model_type,
    model_version=model.model_version,
    model_object=results['final_model'],
    metadata={
        "test_metrics": results['test_metrics'],
        "target_columns": model.target_columns,
        "hyperparameters": train_cfg.get(f"{model_type}_params", {}),
    },
    model_dir=model_save_dir
)

print("\n✅ [Step 3] 모든 산출물 저장 완료")
display(test_pred_df.head())

## 🏁 모델 학습 완료

### ✅ 생성된 산출물
- **`val_predictions.parquet`**: 검증 폴드 예측 결과 (데이터 누수 방지를 위해 03b단계 앙상블 가중치 최적화 전용으로 사용)
- **`test_predictions.parquet`**: 테스트 폴드 예측 결과 (최종 성능 평가용)
- **Model Artifacts**: `.pkl` 파일 및 `registry.json` 메타데이터

### 🔁 다음 단계
- **단일 모델**로 미래 예측을 바로 수행하려면 **`04_forecast_future.ipynb`**로 이동하십시오.
- 여러 모델을 결합하여 예측 성능을 향상시키려면 `config.yaml`에 `active_model: lgbm+rf` 등을 설정한 후 **`03b_train_ensemble.ipynb`**를 실행하십시오.